# 00 — Dataset explorationWhat we are actually training on, and why the headline count of 2,231 ismisleading.

In [ ]:
import sys, csv, collectionssys.path.insert(0, "..")import numpy as npimport matplotlib.pyplot as pltfrom PIL import Imagefrom src.config import processed_dir, DataConfigfrom src.data.dataset import PokemonArtwork, TYPES

## Load the manifest

In [ ]:
root = processed_dir(256, tight_crop=True)with (root / "manifest.csv").open(encoding="utf-8") as fh:    rows = list(csv.DictReader(fh))kept = [r for r in rows if r["kept"] == "1"]print(f"processed : {len(rows)}")print(f"kept      : {len(kept)}")print(f"  normal  : {sum(1 for r in kept if r['shiny'] == '0')}")print(f"  shiny   : {sum(1 for r in kept if r['shiny'] == '1')}")print(f"dropped   : {len(rows) - len(kept)} near-duplicates")

## The number that mattersShinies are recolours. They add palette diversity and **zero shape diversity**,so for anything to do with mode coverage or memorisation the effective datasetis the count of distinct *shapes*, not files.

In [ ]:
structural = sum(1 for r in kept if r["shiny"] == "0")print(f"files                : {len(kept)}")print(f"distinct shapes      : {structural}")print(f"inflation from shiny : {len(kept)/structural:.2f}x")

## Coverage by generation

In [ ]:
gens = collections.Counter(r["generation"] or "form" for r in kept)order = [str(i) for i in range(1, 10)] + ["form"]counts = [gens.get(g, 0) for g in order]fig, ax = plt.subplots(figsize=(9, 3.2))ax.bar(range(len(order)), counts, color=["#4c78a8"]*9 + ["#bab0ac"])ax.set_xticks(range(len(order)))ax.set_xticklabels([f"Gen {g}" if g != "form" else "forms" for g in order], rotation=45, ha="right")ax.set_ylabel("images"); ax.set_title("Coverage by generation")for i, c in enumerate(counts):    ax.text(i, c + 4, str(c), ha="center", fontsize=8)ax.spines[["top", "right"]].set_visible(False)plt.tight_layout(); plt.show()print("Gen 9 present:", gens.get("9", 0))

## Type distributionCollected in Phase 0 for the stretch-goal conditional model. Note how thin eachtype is — this is the evidence for the honest warning that type-conditionalgeneration is data-starved.

In [ ]:
types = collections.Counter()for r in kept:    for t in filter(None, r["types"].split("|")):        types[t] += 1names, vals = zip(*sorted(types.items(), key=lambda kv: -kv[1]))fig, ax = plt.subplots(figsize=(9, 3.2))ax.bar(names, vals, color="#4c78a8")ax.set_xticklabels(names, rotation=45, ha="right")ax.set_ylabel("images"); ax.set_title("Images per type (multi-label)")ax.spines[["top", "right"]].set_visible(False)plt.tight_layout(); plt.show()structural_per_type = {t: v for t, v in types.items()}print(f"median images/type: {np.median(list(vals)):.0f}")print("-> ~70 distinct shapes per type. Genuinely data-starved for conditioning.")

## What the images look like

In [ ]:
ds = PokemonArtwork(resolution=256, mirror=False)idx = np.random.default_rng(7).choice(len(ds), 32, replace=False)fig, axes = plt.subplots(4, 8, figsize=(14, 7))for ax, i in zip(axes.ravel(), idx):    img = (ds[int(i)].permute(1, 2, 0).numpy() + 1) / 2    ax.imshow(img); ax.axis("off")plt.suptitle("Preprocessed training set (256x256, tight-cropped, white ground)")plt.tight_layout(); plt.show()

## Why this is a hard few-shot problemFastGAN's showcase results are on *homogeneous* sets — shells, portraits, asingle art style. Pokémon are structurally wild: birds, blobs, machines,dragons. Comparing the pixel variance of the set against a same-sized sample ofone visual category makes the point.

In [ ]:
sample = np.stack([ds[int(i)].numpy() for i in idx])print(f"per-pixel std across the sample : {sample.std():.3f}")print(f"mean image contrast              : {sample.mean(axis=0).std():.3f}")mean_img = (sample.mean(axis=0).transpose(1, 2, 0) + 1) / 2fig, ax = plt.subplots(figsize=(3.2, 3.2))ax.imshow(np.clip(mean_img, 0, 1)); ax.axis("off")ax.set_title("Mean image", fontsize=10)plt.show()print("A nearly featureless blur -- there is no canonical Pokemon pose or silhouette")print("to anchor on, unlike faces (FFHQ) or animal heads (AFHQ).")

## Takeaways1. **~1,308 distinct shapes**, not 2,231. Reason about the smaller number.2. **Gen 1–9 all covered**, including 199 Gen-9 base entries plus forms.3. **Structurally diverse**, which makes this materially harder than the   few-shot benchmarks FastGAN was demonstrated on.4. ~70 shapes per type — conditional generation (Phase 6) is data-starved and   should be attempted with coarse type groups first.